# Lab A — 用 `adk eval` 評保健品 agent（Colab · API key 版）

延續你蓋的**保健品文案小組**，這一關**回頭評它**：走對流程沒（trajectory）＋ 產出夠好沒（response）。

> **認證用 AI Studio API key**（不是 Vertex）：因為公司政策擋「第三方 notebook 用你的 Google 帳號存取 GCP」。
> API key 只是一串字、不走你的帳號 OAuth → 繞過封鎖。**Runtime → Run all** 就好。

## 0. 安裝（google-adk 的 `[eval]` 提供 adk eval）

In [ ]:
!pip install -q "google-adk[eval]==1.37.0"

## 1. 提供 AI Studio API key（用 Colab Secrets，較安全）

**建議做法（key 不會進 notebook/輸出/git）**：
1. 左側 **🔑（Secrets）圖示** → **Add new secret**
2. Name 填 `GOOGLE_API_KEY`、Value 貼上 key
3. 打開該 secret 的 **Notebook access** 開關

> 沒設 secret 也沒關係——執行下面那格會自動改用輸入框讓你貼（一樣不會存進 notebook）。
> ⚠️ key 課後記得去 AI Studio 刪掉。

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("已從 Colab Secrets 讀到 API key ✅")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Colab Secrets 沒設 → 直接貼 API key（不顯示）：")
    print("已用輸入框設定 API key ✅")

# 確保走 AI Studio Developer API，不走 Vertex
os.environ.pop("GOOGLE_GENAI_USE_VERTEXAI", None)

## 2. 拿 agent 程式碼（clone repo）＋寫 .env

clone 後 `lab2_multi_agent/`（agent）和 `lab_eval/`（考卷、門檻）就都在了。

In [ ]:
!git clone -q https://github.com/amelielee-tech/adk-workshop.git

# adk eval 會讀 repo 根目錄的 .env → 給它 API key（不設 USE_VERTEXAI，就走 AI Studio）
with open("adk-workshop/.env", "w") as f:
    f.write(f"GOOGLE_API_KEY={os.environ['GOOGLE_API_KEY']}\n")

print("repo cloned；.env 寫好（用 API key）")
!ls adk-workshop/lab_eval/

## 3. 跑 `adk eval`（評 lab2_multi_agent）

一行指令：`adk eval` **〈評誰〉〈用哪份考卷〉** `--config`**〈及格線〉**。
會真的把 agent 跑兩次（魚油、益生菌），約 1–2 分鐘。

In [ ]:
!cd adk-workshop && adk eval lab2_multi_agent lab_eval/copy_agent.evalset.json \
  --config_file_path lab_eval/test_config.json 2>&1 | grep -E "Tests passed|Tests failed|Using evaluation" 

## 4. 讀成乾淨的分數表（兩軸分開看）

In [ ]:
import json, glob
import pandas as pd

rows = []
for f in sorted(glob.glob("adk-workshop/lab2_multi_agent/.adk/eval_history/*.json")):
    d = json.load(open(f))
    for c in d["eval_case_results"]:
        r = {"case": c["eval_id"]}
        for m in c["overall_eval_metric_results"]:
            r[m["metric_name"]] = round(m["score"], 3)
            r[m["metric_name"] + " → "] = "PASS" if m["eval_status"] == 1 else "FAIL"
        rows.append(r)

pd.DataFrame(rows)

**怎麼讀**：
- `tool_trajectory_avg_score`＝走對流程沒。門檻 1.0、用 IN_ORDER。
- `response_match_score`＝文案字面（ROUGE-1 F1）像不像。**中文創作型天生低又飄**（門檻只 0.15）——這正是下一關 Lab B 改用 LLM-judge 的理由。

## 5. 效率：打一次 agent 看 latency + token

In [ ]:
!cd adk-workshop && python lab_eval/probe_efficiency.py

## 收尾

- **agent 評估分兩軸**：走對流程（trajectory）＋ 產出夠好（response）。
- **每個指標都有極限**：ROUGE 對創作型中文很弱 → 所以 Lab B 用 LLM-judge。
- **效率**（latency/token）只有真的跑 agent 才量得到。

一句話：**eval 讓「感覺不錯」變成「量得出來」。**